In [2]:
#from google.colab import drive
#drive.mount('/content/drive')

In [5]:
!pip install transformers

Looking in indexes: https://pypi.org/simple, https://packagecloud.io/github/git-lfs/pypi/simple
You should consider upgrading via the 'pip install --upgrade pip' command.


In [6]:
import torch
from transformers import RobertaForSequenceClassification, RobertaTokenizer
import numpy as np

To use data.metrics please install scikit-learn. See https://scikit-learn.org/stable/index.html


In [7]:
if torch.cuda.is_available():
  device = torch.device('cuda')
else:
  device = torch.device('cpu')
device

device(type='cpu')

# Drivers

In [9]:
MAX_LEN = 64

In [20]:
#directory = '/content/drive/My Drive/MY PYTORCH MODELS/Distil RoBERTa (Sequence Classification on CoLA)/my_model_save/'
directory = './my_model_save/'

In [21]:
query = ['Mike and Morris lived in the same village.',
 'While Morris owned the largest jewelry shop in the village, Mike was a poor farmer.',
 'Both had large families with many sons, daughters-in-law and grandchildren.',
 'One fine day, Mike, tired of not being able to fed his family, decided to leave the village and move to the city where he was certain to earn enough to feed everyone.',
 'Along with his family, he left the village for the city.',
 'At night, they stopped under a large tree.',
 'There was a stream running nearby where they could freshen up themselves.',
 'He told his sons to clear the area below the tree, he told his wife to fetch water and he instructed his daughters-in-law to make up the fire and started cutting wood from the tree himself.',
 'They didn’t knew that in the branches of the tree, there was a thief hiding.',
 'He watched as Mike’s family worked together and also noticed that they had nothing to cook.',
 'Mike’s wife also thought the same and asked her husband ” Everything is ready but what shall we eat ?',
 '” Mike raised his hands to heaven and said ” Don’t worry.',
 'He is watching all this from above.',
 'He will help us”.',
 'The thief got worried as he had seen that the family was large and worked well together.',
 'Taking advantage of the fact that they did not know he was hiding in the branches, he decided to make a quick escape.',
 'He climbed down safely when they were not looking and ran for his life.',
 'But, he left behind the bundle of stolen jewels and money which dropped into Mike’s lap.',
 'Mike opened it and jumped with joy when he saw the contents.',
 'The family gathered all their belongings and returned to the village.',
 'There was great excitement when they told everyone how they got rich.',
 'Morris thought that the tree was miraculous and this was a nice and quick way to earn some money.',
 'He ordered his family to pack some clothes and they set off as if on a journey.',
 'They also stopped under the same tree and Morris started commanding everyone as Mike had done.',
 'But no one in his family was willing to obey his orders.',
 'Being a rich family, they were used to having servants all around.',
 'So, the one who went to the river to fetch water enjoyed a nice bath.',
 'The one who went to get wood for fire went off to sleep.',
 'Morris’s wife said ” Everything is ready but what shall we eat ?',
 '” Morris raised his hands and said, ” Don’t worry.',
 'He is watching all this from above.',
 'He will help us”.',
 'As soon as he finished saying, the thief jumped down from the tree with a knife in hand.',
 'Seeing him, everyone started running here and there to save their lives.',
 'The thief stole everything they had and Morris and his family had to return to the village empty handed, having lost all their valuables that they had taken with them.']

# Input Preprocessing

In [22]:
def getTokenizer(directory):
  tokenizer = RobertaTokenizer.from_pretrained(directory)
  return tokenizer

In [23]:
def queryEncoder(sentences, tokenizer): 
  input_ids = []
  for each in sentences:
    encoded_each = tokenizer.encode(
        text = each,
        add_special_tokens = True
    )
    input_ids.append(np.array(encoded_each))
  return input_ids

In [24]:
def encodedQueryPadding(input_ids, MAX_LEN):
  padded_input_ids = []
  for index in range(input_ids.shape[0]):
      padded = np.zeros((MAX_LEN,), dtype=np.int64)
      if len(input_ids[index]) < MAX_LEN:
        padded[:len(input_ids[index])] = input_ids[index][:]
        padded_input_ids.append(padded)
      else: 
        padded_input_ids.append(input_ids[index][:MAX_LEN])
  return padded_input_ids

In [25]:
def tokenAttentionMasking(padded_input_ids):
  attention_masks = []
  for index in range(padded_input_ids.shape[0]):
    att_mask = [int(each > 0) for each in padded_input_ids[index]]
    attention_masks.append(att_mask)
  return attention_masks

In [26]:
def inputPreprocessing(query, directory, MAX_LEN):
  tokenizer = getTokenizer(directory)
  input_ids = np.array(queryEncoder(query, tokenizer))
  padded_input_ids = np.array(encodedQueryPadding(input_ids, MAX_LEN))
  attention_masks = np.array(tokenAttentionMasking(padded_input_ids))
  
  return padded_input_ids, attention_masks 

# Getting Model

In [27]:
def getModel(directory, device):
  model = RobertaForSequenceClassification.from_pretrained(directory)
  
  model.to(device)
  for param in model.parameters():
    param.requires_grad = False
  model.eval()
  
  print("model sent to:",device)  
  return model

In [28]:
model = getModel(directory, device)

model sent to: cpu


# Get Predictions

In [29]:
def convertToTensor(obj):
  return torch.tensor(obj)

In [30]:
def getPredictions(model):
  padded_input_ids, attention_masks = inputPreprocessing(query=query, 
                                                       directory=directory, 
                                                       MAX_LEN=MAX_LEN)
  padded_input_ids = convertToTensor(padded_input_ids)
  attention_masks = convertToTensor(attention_masks)
  
  with torch.no_grad():
    pred = model(padded_input_ids.to(device), 
                 attention_mask=attention_masks.to(device), 
                 labels=None)

  logits = pred[0]
  logits = logits.detach().cpu().numpy()
  return logits

In [31]:
def activateLogits(model):
  logit_predictions = getPredictions(model)
  sig_logit = np.array(torch.sigmoid(torch.tensor(logit_predictions)))
  sig_logit_arg = np.argmax(sig_logit, axis=1).flatten()
  sig_logit_arg_value = np.max(sig_logit, axis=1).flatten()
  return sig_logit_arg, sig_logit_arg_value

# Prediction Calls

In [32]:
final_predictions, final_predictions_confidence = activateLogits(model)

In [33]:
print(final_predictions)

[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]


In [34]:
print(final_predictions_confidence)

[0.96824604 0.96151114 0.94794136 0.78092915 0.9551146  0.9362057
 0.94479406 0.9562384  0.7920402  0.94583386 0.89199823 0.9492041
 0.95998883 0.9062085  0.915879   0.9407954  0.9068143  0.9053488
 0.95857733 0.9605048  0.9548514  0.95254546 0.95314336 0.7385399
 0.9568895  0.89893115 0.9422185  0.9278747  0.94277495 0.95167804
 0.95998883 0.9062085  0.934736   0.9193898  0.9182323 ]


# Handling Predictions

In [35]:
prediction_dictionary = {}
for i in range(len(final_predictions)):
  prediction_dictionary[i] = [final_predictions[i],
                              final_predictions_confidence[i]]

In [36]:
print(prediction_dictionary)

{0: [1, 0.96824604], 1: [1, 0.96151114], 2: [1, 0.94794136], 3: [1, 0.78092915], 4: [1, 0.9551146], 5: [1, 0.9362057], 6: [1, 0.94479406], 7: [1, 0.9562384], 8: [1, 0.7920402], 9: [1, 0.94583386], 10: [1, 0.89199823], 11: [1, 0.9492041], 12: [1, 0.95998883], 13: [1, 0.9062085], 14: [1, 0.915879], 15: [1, 0.9407954], 16: [1, 0.9068143], 17: [1, 0.9053488], 18: [1, 0.95857733], 19: [1, 0.9605048], 20: [1, 0.9548514], 21: [1, 0.95254546], 22: [1, 0.95314336], 23: [1, 0.7385399], 24: [1, 0.9568895], 25: [1, 0.89893115], 26: [1, 0.9422185], 27: [1, 0.9278747], 28: [1, 0.94277495], 29: [1, 0.95167804], 30: [1, 0.95998883], 31: [1, 0.9062085], 32: [1, 0.934736], 33: [1, 0.9193898], 34: [1, 0.9182323]}
